In [ ]:
from sql_connection import read_table, write_table, read_query

In [ ]:
df_sum_tot = read_table("ice.sum_subtot")[['elctrnc_est_dtl_id', 'sum_subtot_id', 'tot_amt', 'taxbl_amt', 'tax_tot_amt', 'tot_hr',  'tot_typ_xref_id', 'cieca_tot_subtyp_id',]]

df_sum_typ = read_table("ice.tot_typ_xref")[['tot_typ_xref_id', 'tot_typ_cde', 'cieca_tot_typ_cde', 'cieca_tot_typ_dsc', 'tot_typ_ind', 'est_line_item_typ_id']]

df_subtot_adj = read_table('ice.sum_subtot_adj')[['sum_subtot_id', 'adj_tot_amt']]

In [ ]:
df = df_sum_tot.merge(df_sum_typ, on = "tot_typ_xref_id", how= "left" ).merge(df_subtot_adj, on = "sum_subtot_id", how= "left")

In [ ]:
len(df)

In [ ]:
from pipeline_orchestrator import cast_numeric

In [ ]:
cast_numeric(df, ['tot_amt', 'adj_tot_amt', 'elctrnc_est_dtl_id' ])

In [ ]:
df.columns

In [ ]:
df = df[['est_id', 'sum_subtot_id', 'gross_amt', 'adj_tot_amt', 'tot_amt', 'taxbl_amt',
       'tax_tot_amt', 'tot_hr', 'tot_typ_xref_id', 'cieca_tot_subtyp_id',
       'tot_typ_cde', 'cieca_tot_typ_cde', 'cieca_tot_typ_dsc', 'tot_typ_ind',
       'est_line_item_typ_id']]

In [ ]:
df['gross_amt'] = df['tot_amt'] - df['adj_tot_amt']


In [ ]:
from sql_connection import write_table

write_table(df, table_name= "sum_subtot_master", schema="ice", if_exists= "replace")


In [2]:
test_df = read_table("ice.sum_subtot_master")

print(len(test_df))

print(test_df.columns)

2793032
Index(['est_id', 'sum_subtot_id', 'gross_amt', 'adj_tot_amt', 'tot_amt',
       'taxbl_amt', 'tax_tot_amt', 'tot_hr', 'tot_typ_xref_id',
       'cieca_tot_subtyp_id', 'tot_typ_cde', 'cieca_tot_typ_cde',
       'cieca_tot_typ_dsc', 'tot_typ_ind', 'est_line_item_typ_id'],
      dtype='object')


In [5]:
dd_est_df = read_table("analysis.dd_est_subtot")

est_ids = dd_est_df['est_id'].drop_duplicates().to_list()

print(len(est_ids))

4824


In [6]:
new_df = test_df[test_df['est_id'].isin(est_ids)]

In [ ]:
new_df

In [7]:
write_table(new_df, "dd_est_subtot", "analysis", "replace" )

In [ ]:
df_est = read_query(
    f"""
    SELECT est_id, elctrnc_est_dtl_id
    FROM public.est_cieca_line_master_2
    """
)

df_est = df_est.drop_duplicates(subset=['est_id'])

df_est['elctrnc_est_dtl_id'] = df_est['elctrnc_est_dtl_id'].astype("float64")
df['elctrnc_est_dtl_id'] = df['elctrnc_est_dtl_id'].astype("float64")

df = df.merge(df_est, on= "elctrnc_est_dtl_id", how= "left")

In [ ]:
print(len(df_sum_tot))

print(len(df_sum_typ))

print(len(df_subtot_adj))

In [ ]:
print(df_sum_tot.columns)

print(df_sum_typ.columns)

print(df_subtot_adj.columns)

In [ ]:
subtot_df = read_table('ice.sum_subtot')
subtot_adj_df = read_table('ice.sum_subtot_adj')
subtot_df = read_table('ice.sum_subtot')

In [ ]:
est_img_df = read_table("public.estimate_images_overlap_37k")
est_ids = est_img_df['est_id'].to_list()

In [ ]:
subtot_df = read_table("public.dd_est_subtot")

In [ ]:
su

In [ ]:
subtot_df.columns

In [ ]:
subtot_df = subtot_df[['sum_subtot_id',  'est_id', 'adj_tot_amt', 'tot_amt',]]

In [ ]:
subtot_df

In [ ]:
TABLE_EST_LINE : str = "public.est_valid_cieca_master"
TABLE_SUBTOT   : str = "ice.sum_subtot_master"

valid_df= read_table(TABLE_EST_LINE)

In [ ]:
valid_df['est_id'] = valid_df['est_id'].astype(int)

In [ ]:
valid_df.dtypes

In [ ]:
matching_ids = valid_df[valid_df['est_id'].isin(est_ids)]


In [ ]:
est_ids = matching_ids['est_id'].drop_duplicates().to_list()

len(est_ids)

In [ ]:
write_table(matching_ids, "dd_est_line", schema= "analysis")

In [ ]:
img_path_df = read_table("public.est_img_path")

img_path_df = img_path_df.dropna(subset=['vin', 'licplte_nbr'])

In [ ]:
img_path_df['est_id'] = img_path_df['est_id'].astype(int)

In [ ]:
img_path_df

In [ ]:
est_ids = img_path_df['est_id'].drop_duplicates().to_list()

In [ ]:
img_matching_ids = img_path_df[img_path_df['est_id'].isin(est_ids)]

In [ ]:
img_matching_ids

In [ ]:
write_table(img_matching_ids, "dd_est_imgs", schema= "analysis")

In [ ]:
dd_subtot = read_table("ice.sum_subtot_master")

In [ ]:
dd_subtot_matching = dd_subtot[dd_subtot['est_id'].isin(est_ids)]

In [ ]:
dd_subtot_matching['sum_subtot_id'] = dd_subtot_matching['sum_subtot_id'].astype(int)

In [ ]:
dd_subtot_matching.dtypes

In [ ]:
subtot_adj_df.dtypes

In [ ]:
subtot_adj_df = read_table("ice.sum_subtot_adj")[["sum_subtot_id", "adj_tot_amt"]]
subtot_adj_df["sum_subtot_id"] = subtot_adj_df["sum_subtot_id"].astype(int)
subtot_df = dd_subtot_matching.merge(subtot_adj_df, on="sum_subtot_id", how="left")

In [ ]:
write_table(subtot_df, "dd_est_subtot", schema= "analysis")

In [ ]:
subtot_df.columns